<a href="https://colab.research.google.com/github/nailasalmayusroini/Computer_Vision/blob/main/Assignment02/CVL_Assignment02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Student Identification
Naila Salma Yusroini

24/536726/PA/22766

## Description
This assignment implements a non-parametric computer vision pipeline in pure python to detect barin tumors using the Global Binary thresholding and Centroid Calculation.

In [ ]:
def read_pgm(path):
    """Reads a P5 (binary) PGM file."""
    try:
        with open(path, 'rb') as f:
            header = f.readline().decode().strip()
            if header != 'P5':
                print("Error: File is not a P5 PGM.")
                return None, 0, 0, 0

            # Skip comments
            line = f.readline().decode().strip()
            while line.startswith('#'):
                line = f.readline().decode().strip()

            width, height = map(int, line.split())
            max_val = int(f.readline().decode().strip())
            raw_data = f.read()

        pixels = []
        for y in range(height):
            row = [raw_data[y * width + x] for x in range(width)]
            pixels.append(row)
        return pixels, width, height, max_val
    except FileNotFoundError:
        print(f"Error: File {path} not found.")
        return None, 0, 0, 0

def calculate_stats(pred, gt, w, h):
    """Manual calculation of Accuracy and IoU."""
    tp = tn = fp = fn = 0
    for y in range(h):
        for x in range(w):
            p, g = pred[y][x], gt[y][x]
            if p == 1 and g == 1: tp += 1
            elif p == 0 and g == 0: tn += 1
            elif p == 1 and g == 0: fp += 1
            elif p == 0 and g == 1: fn += 1

    accuracy = (tp + tn) / (w * h)
    iou = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0
    return accuracy, iou

# --- MAIN EXECUTION ---
scan_p = input("Enter path to your brain scan (PGM): ")
gt_p = input("Enter path to ground truth (PGM) [Leave blank if you don't have one]: ")

scan, w, h, _ = read_pgm(scan_p)

if scan:
    # 1. Set the threshold (Adjust this! 0-255)
    # If the brain is visible, try a high number like 200 to isolate the tumor
    thresh = 180
    pred_mask = [[1 if p > thresh else 0 for p in row] for row in scan]

    # 2. Handle Ground Truth
    if gt_p.strip():
        gt_data, _, _, _ = read_pgm(gt_p)
        gt_mask = [[1 if p > 128 else 0 for p in row] for row in gt_data]
    else:
        print("\n[Note] No ground truth provided. Using a dummy comparison.")
        # For demonstration, we'll compare the prediction against a slightly
        # stricter version of itself to simulate an 'ideal' mask.
        gt_mask = [[1 if p > 220 else 0 for p in row] for row in scan]

    # 3. Object Detection (Centroid)
    sx = sy = count = 0
    for y in range(h):
        for x in range(w):
            if pred_mask[y][x] == 1:
                sx += x; sy += y; count += 1

    # 4. Results
    print("\n" + "="*30)
    if count > 0:
        print(f"Tumor Detected at: X={sx//count}, Y={sy//count}")
    else:
        print("Tumor not detected. Try lowering the threshold.")

    acc, iou = calculate_stats(pred_mask, gt_mask, w, h)
    print(f"Accuracy: {acc * 100:.2f}%")
    print(f"IoU Score: {iou:.4f}")
    print("="*30)

Enter path to your brain scan (PGM): /content/tumor_scan.pgm
Enter path to ground truth (PGM) [Leave blank if you don't have one]: /content/tumor_mask.pgm

Tumor Detected at: X=161, Y=234
Accuracy: 96.00%
IoU Score: 0.3179


## Process
The process begins with Global thresholding, where the computer scans the raw P5 PGM file and labels any pixel brighter than a set value (180) as "tumor" and everything else as "healthy."

This is done because tumors often appear as bright, high-intensity spots in MRI scans. After isolating these bright pixels, the system caculates their Centroids, or center of mass, by averaging their coordinates. This allows the code to pinpoints the tumor's exact location. Which in this case, is at (161, 234).

## Result Analysis
To see how well the code performed, we compare it's "guess" againts a hand-drawn Ground Truth mask using two main metrics, which is the Accuracy and the Intersection over Union (IoU).

The assignment then achieved a high 96.00% accuracy, which shows that the code is very good at identifying healthy brain tissue. Besides that, the IoU give a result of 0.3179, which in this case is the most important result. It proves that the computer's detection actually overlaps with the real tumor.

The reason why the IoU isn't perfect is because the computer only selects the absolute brightest pixels, which means bright areas like the skull or dense bone can have the computer mistakenly label it as a tumor since they share the same high intensity. These 'False Positives' increase the total area detected by the computer without increasing the actual overlap, which natually lowers the final IoU score even if the tumor itself was correctly identified. Overall, the results show that even a simple mathematical rule can successfully find the center of a tumor with high reliability.